# Strategy D: TabPFN (Prior-Data Fitted Network) Benchmark
### Foundation Transformer for Small-Sample Tabular Screening Without Overfitting

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

---

### Why Strategy D (TabPFN)?
1. **Zero Visual Noise**: Bypasses the broken heuristic word-cropping and aspect-ratio distortion of ViTs. Operates directly on the student's complete session statistics (stroke fragmentation, baseline wandering, inter-word spacing fatigue, pen pressure faint rate, Devanagari headline continuity).
2. **Pre-Trained on Synthetic Priors**: TabPFN was trained on millions of synthetic structural causal models. It delivers instantaneous Bayesian inference in $< 1$ second and has theoretical immunity to overfitting on small cohort sizes ($N \le 1,000$).
3. **Cohort-Wide Evaluation**: Strictly evaluated across all 215 students (175 Control, 40 Dysgraphic) using the locked 5-repeat $\times$ 5-fold Stratified Group Cross-Validation split.


In [ ]:
# 1. Install TabPFN and Required Packages
!pip install -q tabpfn scikit-learn pandas numpy matplotlib seaborn tqdm lightgbm

import torch
import tabpfn
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"TabPFN Inference Device: {device}")


In [ ]:
# 2. Ingest Multi-School Student-Level Clinical Dataset (215 Students x 406 Features)
import pandas as pd
from pathlib import Path

DATA_FILE = "dataset_student_level.csv"

# Download directly from GitHub if not present
if not Path(DATA_FILE).exists():
    print("Fetching dataset_student_level.csv (1.6 MB)...")
    !curl -L -o dataset_student_level.csv https://raw.githubusercontent.com/Sparrow375/Dysgraphia-Detection/main/data/datasets/dataset_student_level.csv
    print("Download complete!")

df = pd.read_csv(DATA_FILE)
print(f"Cohort Size: {len(df)} students")
print(f"Class Distribution: {df['label'].value_counts().to_dict()} (0=Control, 1=Dysgraphia)")
print(f"School Distribution: {df['school'].value_counts().to_dict()}")
df.head(3)


In [ ]:
# 3. TabPFN Token Setup (If Using TabPFN v2)
import os

# Enter your free TabPFN API Key from https://ux.priorlabs.ai/account (takes 30s to register)
# If left blank, TabPFN will prompt interactively on first run
TABPFN_TOKEN = "" # @param {type:"string"}

if TABPFN_TOKEN:
    os.environ["TABPFN_TOKEN"] = TABPFN_TOKEN
    print("TABPFN_TOKEN configured successfully!")
else:
    print("No token entered. TabPFN will prompt for interactive login if required.")


In [ ]:
# 4. Locked 5-Repeat x 5-Fold Cross-Validation: TabPFN vs Baselines
import numpy as np
from tabpfn import TabPFNClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, roc_curve

FEATURE_SETS = {
    "Curated_5": [
        "components_per_unit_width_max",
        "components_per_unit_width_std",
        "components_per_unit_width_eng_minus_hindi",
        "matra_ratio_max",
        "shirorekha_rms_deviation_norm_max",
    ],
    "Clinical_Top15": [
        "gap_fraction_above_2h_late_minus_early",
        "kappa_sign_changes_per_h_iqr",
        "pressure_faint_rate_tool_norm",
        "baseline_rmse_norm_std",
        "interaction_pen_bloat",
        "component_height_cv_iqr",
        "components_per_unit_width_iqr",
        "erasure_patch_ratio",
        "slant_mean_deg_std",
        "gap_fraction_above_2h_std",
        "smudge_halo_darkness_max",
        "pressure_intensity_cv_median",
        "baseline_slope_mean_late_minus_early",
        "tangent_variance_short_wavelength_iqr",
        "gap_cv_p90",
    ],
}

def resolve_cols(cols):
    res = []
    for c in cols:
        if c in df.columns:
            res.append(c)
        elif f"z_{c}" in df.columns:
            res.append(f"z_{c}")
    return res

y_true = df["label"].values.astype(int)
benchmark_results = []
oof_predictions = {}

for fset_name, raw_cols in FEATURE_SETS.items():
    active_cols = resolve_cols(raw_cols)
    X = df[active_cols].fillna(df[active_cols].median()).values
    print(f"\n{'='*70}\nBENCHMARKING: {fset_name} ({len(active_cols)} features)\n{'='*70}")

    # A. Elastic-Net Logistic Regression
    rep_aucs_en = []
    oof_en = np.zeros(len(df))
    for r in range(5):
        for f in range(5):
            tr = (df[f"repeat_{r}_fold"] != f).values
            val = (df[f"repeat_{r}_fold"] == f).values
            clf = LogisticRegression(penalty="elasticnet", solver="saga", C=0.10, l1_ratio=0.30, class_weight="balanced", max_iter=2000, random_state=42)
            clf.fit(X[tr], y_true[tr])
            oof_en[val] += clf.predict_proba(X[val])[:, 1] / 5.0
        rep_aucs_en.append(roc_auc_score(y_true, oof_en))
    auc_en = roc_auc_score(y_true, oof_en)
    pr_en = average_precision_score(y_true, oof_en)
    print(f"  • Elastic-Net LR     : AUROC = {auc_en:.4f} | AUPRC = {pr_en:.4f}")

    # B. TabPFN Classifier
    try:
        rep_aucs_tab = []
        oof_tab = np.zeros(len(df))
        for r in range(5):
            for f in range(5):
                tr = (df[f"repeat_{r}_fold"] != f).values
                val = (df[f"repeat_{r}_fold"] == f).values
                clf = TabPFNClassifier(device=device)
                clf.fit(X[tr], y_true[tr])
                oof_tab[val] += clf.predict_proba(X[val])[:, 1] / 5.0
            rep_aucs_tab.append(roc_auc_score(y_true, oof_tab))

        auc_tab = roc_auc_score(y_true, oof_tab)
        pr_tab = average_precision_score(y_true, oof_tab)
        fpr, tpr, _ = roc_curve(y_true, oof_tab)
        sens_80 = tpr[np.where(fpr <= 0.20)[0][-1]]
        sens_90 = tpr[np.where(fpr <= 0.10)[0][-1]]

        print(f"  • TabPFN Foundation  : AUROC = {auc_tab:.4f} | AUPRC = {pr_tab:.4f} | Sens@80% = {sens_80*100:.1f}% | Sens@90% = {sens_90*100:.1f}%")
        oof_predictions[fset_name] = oof_tab
        benchmark_results.append({
            "feature_set": fset_name,
            "auroc": auc_tab,
            "auprc": pr_tab,
            "sens_80": sens_80,
            "sens_90": sens_90
        })
    except Exception as e:
        print(f"  [!] TabPFN Error: {e}")


In [ ]:
# 5. Diagnostic ROC & PR Curves for TabPFN
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), dpi=120)

for fset_name, probs in oof_predictions.items():
    fpr, tpr, _ = roc_curve(y_true, probs)
    auc_val = roc_auc_score(y_true, probs)
    ax1.plot(fpr, tpr, lw=2.5, label=f'TabPFN ({fset_name}: AUROC={auc_val:.4f})')

    prec, rec, _ = precision_recall_curve(y_true, probs)
    pr_val = average_precision_score(y_true, probs)
    ax2.plot(rec, prec, lw=2.5, label=f'TabPFN ({fset_name}: AUPRC={pr_val:.4f})')

ax1.plot([0, 1], [0, 1], color='gray', linestyle='--')
ax1.axvline(0.10, color='crimson', linestyle=':', label='90% Specificity Operating Point')
ax1.set_xlabel('False Positive Rate (1 - Specificity)')
ax1.set_ylabel('True Positive Rate (Sensitivity)')
ax1.set_title('TabPFN ROC Curves (215 Students)', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

no_skill = (y_true == 1).sum() / len(y_true)
ax2.axhline(no_skill, color='gray', linestyle='--', label=f'Baseline Chance ({no_skill:.2f})')
ax2.set_xlabel('Recall (Sensitivity)')
ax2.set_ylabel('Precision (Positive Predictive Value)')
ax2.set_title('TabPFN Precision-Recall Curves', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# 6. Export TabPFN Out-of-Fold Predictions
export_df = pd.DataFrame({
    "student_id": df["student_id"],
    "school": df["school"],
    "grade": df["grade"],
    "label": df["label"],
})

for fset_name, probs in oof_predictions.items():
    export_df[f"prob_tabpfn_{fset_name.lower()}"] = probs

export_path = "p_tabpfn_predictions.csv"
export_df.to_csv(export_path, index=False)
print(f"Exported TabPFN predictions to: {export_path} ({len(export_df)} students)")

try:
    from google.colab import files
    files.download(export_path)
except Exception:
    pass
